# Robust Relay Coordination Model
Este notebook está preparado para ejecutarse en Google Colab. Sigue el orden de las celdas para instalar el solver, cargar tus datos y optimizar el sistema.

### 1. Instalación de Dependencias
Se instalará Pyomo y la librería `highspy` que contiene el motor matemático de HiGHS, altamente compatible con Colab.

In [ ]:
!pip install -q pyomo pandas numpy matplotlib highspy
print("✅ Dependencias instaladas correctamente.")

### 2. Carga de Datos
Ejecuta esta celda y usa el botón **'Elegir archivos'** para subir tu archivo `beta_df.csv`.

In [ ]:
import os
from google.colab import files

if not os.path.exists('beta_df.csv'):
    print("Por favor, sube tu archivo 'beta_df.csv':")
    uploaded = files.upload()
    if 'beta_df.csv' in uploaded:
        print("✅ Archivo cargado exitosamente.")
else:
    print("✅ El archivo 'beta_df.csv' ya se encuentra en el entorno.")

### 3. Modelo Matemático y Optimización
El script procesará las curvas, construirá las restricciones robustas y resolverá el problema usando `appsi_highs`.

In [ ]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo

# =====================================================================
# 1. DATA LOADING AND PROCESSING
# =====================================================================
print("--- Starting data processing ---")
csv_path = 'beta_df.csv'
try:
    beta_df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"File {csv_path} not found. Ensure it was uploaded correctly.")
    raise

# Filter out negative Beta values
filtered_df = beta_df[beta_df['Beta'] >= 0]

# System Sets Definition
R = ['R1', 'R2', 'R3']
C = ['STI', 'SI'] # Add 'VI', 'EI' if they exist in your actual CSV dataset
P = [('R2', 'R1'), ('R3', 'R1'), ('R3', 'R2')] # (Principal relay i, Backup relay j)
F = 'F1' # Target fault for coordination

# Dictionaries to store extracted parameters
k1_dict = {}
k2_dict = {}
beta0_dict = {}  # Nominal (mean) value of beta_jq
betaL_dict = {}  # Uncertainty limit (e.g., maximum absolute deviation)
beta_obj_dict = {} # Mean betas for the objective function

# EXTRACTION OF k1, k2 AND NOMINAL PARAMETERS
for i, j in P:
    for c in C:
        for q in C:
            df_prin = filtered_df[(filtered_df['Rele'] == i) & (filtered_df['Curva'] == c) & (filtered_df['Falla'] == F)]
            df_back = filtered_df[(filtered_df['Rele'] == j) & (filtered_df['Curva'] == q) & (filtered_df['Falla'] == F)]
            
            df_merged = pd.merge(df_prin, df_back, on='Distancia', suffixes=('_prin', '_back')).sort_values(by='Distancia')
            
            y = df_merged['Beta_prin'].to_numpy()
            x = df_merged['Beta_back'].to_numpy()
            
            if len(x) >= 2:
                k2, k1 = np.polyfit(x, y, 1)
            else:
                k2, k1 = 1.0, 0.0
                
            k1_dict[(i, j, q, c)] = k1
            k2_dict[(i, j, q, c)] = k2
            
            if (j, q) not in beta0_dict:
                if len(x) > 0:
                    b0 = np.mean(x)
                    bl = np.max(np.abs(x - b0))
                    if bl == 0:
                        bl = b0 * 0.01 
                else:
                    b0 = 1.0
                    bl = 0.1
                
                beta0_dict[(j, q)] = b0
                betaL_dict[(j, q)] = bl
                
for i in R:
    for c in C:
        df_obj = filtered_df[(filtered_df['Rele'] == i) & (filtered_df['Curva'] == c)]
        beta_obj_dict[(i, c)] = df_obj['Beta'].mean() if not df_obj.empty else 1.0

# =====================================================================
# 2. PYOMO ROBUST MODEL CONSTRUCTION
# =====================================================================
print("--- Building Robust Model in Pyomo ---")

model = pyo.ConcreteModel(name="Robust_Coordination_V2")

model.R = pyo.Set(initialize=R)
model.C = pyo.Set(initialize=C)
model.P = pyo.Set(dimen=2, initialize=P)

model.CTI = pyo.Param(initialize=0.2)
model.M   = pyo.Param(initialize=1000.0)
model.y_lb = pyo.Param(initialize=0.0005)
model.y_ub = pyo.Param(initialize=1.2)

C_index = {"STI": 0.0, "SI": 0.0, "VI": 0.0, "EI": 0.0} 
def c_param_init(model, c):
    return C_index.get(c, 0.0)
model.C_param = pyo.Param(model.C, initialize=c_param_init)

model.k1    = pyo.Param(model.P, model.C, model.C, initialize=k1_dict)
model.k2    = pyo.Param(model.P, model.C, model.C, initialize=k2_dict)
model.beta0 = pyo.Param(model.R, model.C, initialize=beta0_dict, default=1.0)
model.betaL = pyo.Param(model.R, model.C, initialize=betaL_dict, default=0.1)
model.beta_obj = pyo.Param(model.R, model.C, initialize=beta_obj_dict)

model.y = pyo.Var(model.R, model.C, domain=pyo.NonNegativeReals)
model.x = pyo.Var(model.R, model.C, domain=pyo.Binary)
model.z = pyo.Var(model.P, model.C, model.C, domain=pyo.NonNegativeReals)

def obj_rule(model):
    return sum(model.beta_obj[i, c] * model.y[i, c] + model.x[i, c] * model.C_param[c] 
               for i in model.R for c in model.C)
model.objective = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

def single_curve_rule(model, i):
    return sum(model.x[i, c] for c in model.C) == 1
model.single_curve = pyo.Constraint(model.R, rule=single_curve_rule)

def tms_lower_rule(model, i, c):
    return model.y[i, c] >= model.y_lb * model.x[i, c]
model.tms_lower = pyo.Constraint(model.R, model.C, rule=tms_lower_rule)

def tms_upper_rule(model, i, c):
    return model.y[i, c] <= model.y_ub * model.x[i, c]
model.tms_upper = pyo.Constraint(model.R, model.C, rule=tms_upper_rule)

def rc_base_rule(model, i, j, q, c):
    nominal_part = (model.k1[i,j,q,c] * model.y[i,c] + 
                    model.k2[i,j,q,c] * model.beta0[j,q] * model.y[i,c] - 
                    model.beta0[j,q] * model.y[j,q] + 
                    model.C_param[c] - model.C_param[q] + model.CTI)
    return nominal_part + model.z[i,j,q,c] <= model.M * (2 - model.x[j,q] - model.x[i,c])
model.rc_base = pyo.Constraint(model.P, model.C, model.C, rule=rc_base_rule)

def rc_z_pos_rule(model, i, j, q, c):
    uncertainty = model.k2[i,j,q,c] * model.betaL[j,q] * model.y[i,c] - model.betaL[j,q] * model.y[j,q]
    return model.z[i,j,q,c] >= uncertainty - model.M * (2 - model.x[j,q] - model.x[i,c])
model.rc_z_pos = pyo.Constraint(model.P, model.C, model.C, rule=rc_z_pos_rule)

def rc_z_neg_rule(model, i, j, q, c):
    uncertainty = model.k2[i,j,q,c] * model.betaL[j,q] * model.y[i,c] - model.betaL[j,q] * model.y[j,q]
    return model.z[i,j,q,c] >= -uncertainty - model.M * (2 - model.x[j,q] - model.x[i,c])
model.rc_z_neg = pyo.Constraint(model.P, model.C, model.C, rule=rc_z_neg_rule)

# =====================================================================
# 3. MODEL RESOLUTION
# =====================================================================
print("--- Starting Optimization ---")
try:
    # Using appsi_highs directly, fully compatible with Colab Linux environment
    solver = pyo.SolverFactory('appsi_highs')
    
    results = solver.solve(model, tee=True)
    
    print("\nOptimization Status:", results.solver.status)
    print("Termination Condition:", results.solver.termination_condition)
    
    print("\n--- Optimal Results ---")
    for i in model.R:
        for c in model.C:
            if pyo.value(model.x[i, c]) > 0.5:
                print(f"Relay {i} -> Selected Curve: {c} | TMS (y) = {pyo.value(model.y[i, c]):.4f}")

except Exception as e:
    print(f"\nCould not solve the model. Error details: {e}")
